In [ ]:
!apt-get install -y ffmpeg 2>/dev/null
!pip install -q yt-dlp requests groq
print("✅ FFmpeg + yt-dlp + Groq ready")

In [ ]:
import os, json, subprocess, re, requests, builtins
from IPython.display import Video, display
from google.colab import files
from groq import Groq

CHANNEL_URL = "https://www.youtube.com/@samir-lail.mestapha-lherda"
FB_PAGE_ID = "1179824185206026"
FB_ACCESS_TOKEN = "<your-facebook-token>"
GROQ_KEY = "<your-groq-api-key>"
client = Groq(api_key=GROQ_KEY)

!wget -q -O bg.jpg "https://images.unsplash.com/photo-1598488035139-bdbb2231ce04?w=1280&h=720&fit=crop"
!wget -q -O bg_music.mp3 "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3"
print("✅ bg.jpg + bg_music.mp3 + config")

In [ ]:
STATE_FILE = "state.json"
if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f: state = json.load(f)
else:
    state = {"downloaded": []}

def save_state():
    with open(STATE_FILE, "w") as f: json.dump(state, f)

def get_latest():
    cmd = ["yt-dlp", "--flat-playlist", "--dump-json",
           "--playlist-end", "5", CHANNEL_URL]
    r = subprocess.run(cmd, capture_output=True, text=True)
    for line in r.stdout.strip().split("\n"):
        if not line: continue
        try:
            data = json.loads(line)
            return {"id": data["id"], "title": data["title"], "url": data["webpage_url"]}
        except: continue
    return None

def download_audio(url, path="input.mp4"):
    # audio-only أسرع بكثير (محتاجين الصوت فقط)
    subprocess.run(["yt-dlp", "-f", "bestaudio[ext=m4a]", "-o", path,
                    "--no-progress", url], check=True, capture_output=True)
    return path

builtins.ok = False
builtins.original_title = ""

print("🔍 أحدث فيديو...")
video = get_latest()
if not video:
    print("❌ ما لقيناش فيديو")
elif video["id"] in state["downloaded"]:
    print(f"⏭️ {video['title'][:40]} - محمل قبل")
else:
    print(f"📥 تحميل صوت: {video['title'][:50]}")
    download_audio(video["url"])
    state["downloaded"].append(video["id"])
    save_state()
    builtins.ok = True
    builtins.original_title = video["title"]
    print(f"✅ {video['title'][:50]}")

In [ ]:
builtins.generated_title = f"🎙️ podcaste abdo | {original_title}" if ok else ""
builtins.generated_desc = f"{original_title}\n—\n#podcast #shorts #podcasteabdo" if ok else ""

if ok:
    try:
        chat = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system",
                 "content": "أنت منتج بودكاست محترف. أعد كتابة عنوان ووصف الفيديو بصياغة جديدة جذابة مع الحفاظ على نفس الفكرة والمحتوى."
                },
                {"role": "user",
                 "content": f"العنوان الأصلي: {original_title}\n\nأعد كتابة:\n1. عنوان جديد للبودكاست يبدأ بـ '🎙️ podcaste abdo |'\n2. وصف جديد (نفس القصة/الفكرة، صياغة مختلفة احترافية)\n3. هاشتاغات: #podcast #shorts #podcasteabdo"
                }
            ]
        )
        raw = chat.choices[0].message.content
        print(raw)

        for line in raw.strip().split("\n"):
            line = line.strip().strip('"').strip("'")
            if any(m in line for m in ["🎙️", "podcaste", "podcast ", "بودكاست"]):
                if "|" in line or "🎙️" in line:
                    generated_title = line.rstrip(",")
            elif line.startswith("#"):
                if line not in generated_desc:
                    generated_desc += f"\n{line}"

        builtins.generated_title = generated_title
        builtins.generated_desc = generated_desc

        import re
        safe = re.sub(r"['\"\\]", "", generated_title)[:60]
        builtins.safe_title = safe
        print(f"\n📝 {safe}")
    except Exception as e:
        print(f"⚠️ Groq error: {e}")
else:
    print("⚠️ ما تحملش فيديو")

In [ ]:
def make_podcast(input_video="input.mp4", output="podcast_episode.mp4",
                 bg="bg.jpg", music="bg_music.mp3", title="بودكاست"):
    filt = (
        f"[0:a]asplit=2[a_fx][a_waves];"
        f"[a_fx]silenceremove=1:0:-50dB,"
        f"asetrate=44100*1.03,atempo=1/1.03,"  # pitch +3%
        f"atempo=1.05,"                        # speed +5%
        f"equalizer=f=100:t=q:w=1:g=5,"        # Bass
        f"equalizer=f=8000:t=q:w=1:g=-3,"      # Treble
        f"aecho=0.8:0.7:60:0.4[voice];"         # Reverb
        f"[voice][1:a]amix=inputs=2:duration=first:weights=1 0.1[a_out];"  # +music
        f"[a_waves]showwaves=s=1280x540:mode=cline:rate=25:colors=#FFD700|white[waves];"
        f"[2:v]scale=1280:720[bg];"
        f"[bg][waves]overlay=format=auto:0:((H-h)/2)[v];"
        f"[v]drawtext=text='podcaste abdo':fontsize=32:fontcolor=white:"
        f"x=(w-text_w)/2:y=H-100:shadowy=2:shadowcolor=black@0.5,"
        f"drawtext=text='{title}':fontsize=18:fontcolor=#CCCCCC:"
        f"x=(w-text_w)/2:y=H-60:shadowy=1:shadowcolor=black@0.3"
    )
    cmd = ["ffmpeg","-y",
           "-i",input_video,   # 0: audio
           "-i",music,         # 1: background music
           "-i",bg,            # 2: background image
           "-filter_complex",filt,
           "-map","[v]","-map","[a_out]",
           "-c:v","libx264","-preset","fast","-crf","23",
           "-pix_fmt","yuv420p","-c:a","aac","-b:a","192k",
           "-movflags","+faststart","-shortest",output]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:])
        return None
    mb = os.path.getsize(output)/(1024*1024)
    print(f"✅ {output} ({mb:.1f} MB)")
    return output

if ok or os.path.exists("input.mp4"):
    t = safe_title if 'safe_title' in dir() else "بودكاست"
    make_podcast(title=t)
else:
    print("⚠️ مكاينش فيديو")

In [ ]:
if os.path.exists("podcast_episode.mp4"):
    display(Video("podcast_episode.mp4", width=720))
    files.download("podcast_episode.mp4")
else:
    print("⚠️ شغل Cell 5")

In [ ]:
if os.path.exists("podcast_episode.mp4") and FB_ACCESS_TOKEN:
    t = generated_title if ok else f"🎙️ podcaste abdo | بودكاست"
    d = generated_desc if ok else "#podcast #shorts #podcasteabdo"
    print(f"📤 رفع...\n   {t[:70]}")
    r = requests.post(
        f"https://graph.facebook.com/v22.0/{FB_PAGE_ID}/videos",
        files={"source": open("podcast_episode.mp4","rb")},
        data={"title": t[:100], "description": d[:1000], "access_token": FB_ACCESS_TOKEN}
    )
    res = r.json()
    if "id" in res:
        print(f"✅ تم! https://facebook.com/{res['id']}")
        save_state()
    else:
        print("❌:", json.dumps(res, indent=2, ensure_ascii=False))
else:
    print("⚠️ الفيديو مكاينش")